In [ ]:
# %%

# Lets import the packages
import torch
from torchvision import transforms
from LymphoMNIST.LymphoMNIST import LymphoMNIST

import torch.nn as nn
from torchvision import models
from torchsummary import summary


import time
import torch.optim as optim
from torchsummary import summary
from Project import Project
from poutyne.framework import Model
from logger import logging
import datetime, os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# check LymphoMNIST virsion
import LymphoMNIST as info
print(f"LymphoMNIST v{info.__version__} @ {info.HOMEPAGE}")

# %%
from utils import device

import numpy as np
import torchvision.transforms as T
from imgaug import augmenters as iaa
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split


def get_dataloaders(
        train_ds,
        val_ds,
        split=(0.5, 0.5),
        batch_size=64,
        *args, **kwargs):
    """
    This function returns the train, val and test dataloaders.
    """

    # now we want to split the val_ds in validation and test
    lengths = np.array(split) * len(val_ds)
    lengths = lengths.astype(int)
    left = len(val_ds) - lengths.sum()
    # we need to add the different due to float approx to int
    lengths[-1] += left

    val_ds, test_ds = random_split(val_ds, lengths.tolist())
    logging.info(f'Train samples={len(train_ds)}, Validation samples={len(val_ds)}, Test samples={len(test_ds)}')

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, *args, **kwargs)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, *args, **kwargs)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, *args, **kwargs)

    return train_dl, val_dl, test_dl


class FilteredLymphoMNIST(Dataset):
    def __init__(self, original_dataset, labels_to_keep):
        self.original_dataset = original_dataset
        # Filter indices based on labels_to_keep
        self.indices = [i for i, (_, label) in enumerate(original_dataset) if label in labels_to_keep]

    def __getitem__(self, index):
        # Map the current index to the index of the original dataset
        original_index = self.indices[index]
        return self.original_dataset[original_index]

    def __len__(self):
        return len(self.indices)


class ConvertToRGB:
    """
    Convert 1-channel tensors to 3-channel tensors by duplicating the channel 3 times.
    """
    def __call__(self, tensor):
        # Check if the tensor is 1-channel (C, H, W) where C == 1
        if tensor.shape[0] == 1:
            # Duplicate the channel 3 times
            tensor = tensor.repeat(3, 1, 1)
        return tensor





im_size = 64

val_transform = T.Compose([
    T.Resize((im_size, im_size)),
    T.ToTensor(),
    T.Normalize([0.4819], [0.1484]),  # Adjusted for 1-channel
    ConvertToRGB()
])


#%%
# our hyperparameters
params = {
    'lr': 5e-5,
    'batch_size': 16,
    'epochs': 1000,
    'model': "teacher_final"
}


# Initialize dataset
original_train_ds = LymphoMNIST(root='./dataset', train=True, download=True, transform=val_transform, num_classes=3)
original_test_ds = LymphoMNIST(root='./dataset', train=False, download=True, transform=val_transform, num_classes=3)


# Specify labels to keep
labels_to_keep = [0, 1] # 0: B, 1: T4, 2: T8

# Initialize filtered dataset with labels to keep
train_ds = FilteredLymphoMNIST(original_train_ds, labels_to_keep)
test_ds= FilteredLymphoMNIST(original_test_ds, labels_to_keep)


train_dl, val_dl, test_dl = get_dataloaders(train_ds,
                                            test_ds,
                                            split=(0.5, 0.5),
                                            batch_size=params['batch_size'],
                                            num_workers=4
                                           )


In [ ]:
# Load a pre-trained ResNet50 model
resnet50 = models.resnet50(weights='IMAGENET1K_V1')

# Adjust the final fully connected layer for your number of classes
num_ftrs = resnet50.fc.in_features
num_classes = 2  # Change to your number of classes
resnet50.fc = nn.Linear(num_ftrs, num_classes)


# Load the saved model weights
# model_weights_path = 'checkpoint/10 March 10:37-modelteacher_final.pt'
model_weights_path = 'checkpoint/10 March 10:37-modelteacher_final.pt'# Update this path
resnet50.load_state_dict(torch.load(model_weights_path, map_location=device))

# Move the modified model to CUDA
cnn = resnet50.to(device)

# Adjust the input size if necessary
summary(cnn, (3, 64, 64))

In [1]:
import os
import torch
import numpy as np
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, random_split
from torchsummary import summary
import torch.nn as nn
import torch.optim as optim
from logger import logging
from Project import Project
from poutyne.framework import Model
from LymphoMNIST.LymphoMNIST import LymphoMNIST
import LymphoMNIST as info
from utils import device

# Setting CUDA device (should be set according to the system's configuration or dynamically)
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# Check LymphoMNIST version
print(f"LymphoMNIST v{info.__version__} @ {info.HOMEPAGE}")

# Dataset preprocessing and dataloaders
im_size = 64
val_transform = transforms.Compose([
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.4819], [0.1484]),  # Adjusted for 1-channel
    lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x  # Convert 1-channel tensors to 3-channel
])

class FilteredLymphoMNIST(Dataset):
    def __init__(self, original_dataset, labels_to_keep):
        self.original_dataset = original_dataset
        self.indices = [i for i, (_, label) in enumerate(original_dataset) if label in labels_to_keep]

    def __getitem__(self, index):
        return self.original_dataset[self.indices[index]]

    def __len__(self):
        return len(self.indices)

def get_dataloaders(train_ds, val_ds, split=(0.5, 0.5), batch_size=64, **kwargs):
    lengths = np.array(split) * len(val_ds)
    lengths = lengths.astype(int)
    lengths[-1] += len(val_ds) - lengths.sum()  # Adjust for rounding errors
    val_ds, test_ds = random_split(val_ds, lengths.tolist())
    logging.info(f'Train samples={len(train_ds)}, Validation samples={len(val_ds)}, Test samples={len(test_ds)}')

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, **kwargs),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kwargs),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kwargs)
    )

# Model setup
params = {'lr': 5e-5, 'batch_size': 16, 'epochs': 1000, 'model': "teacher_final"}
labels_to_keep = [0, 1]  # Specify labels to keep

# Load datasets
train_ds = FilteredLymphoMNIST(LymphoMNIST(root='./dataset', train=True, download=True, transform=val_transform, num_classes=3), labels_to_keep)
test_ds = FilteredLymphoMNIST(LymphoMNIST(root='./dataset', train=False, download=True, transform=val_transform, num_classes=3), labels_to_keep)

# Initialize dataloaders
train_dl, val_dl, test_dl = get_dataloaders(train_ds, test_ds, split=(0.5, 0.5), batch_size=params['batch_size'], num_workers=4)

# Load and modify a pre-trained ResNet50 model
resnet50 = models.resnet50(pretrained=True)
resnet50.fc = nn.Linear(resnet50.fc.in_features, 2)  # Adjust for the number of classes
model_weights_path = 'checkpoint/10 March 10:37-modelteacher_final.pt'
resnet50.load_state_dict(torch.load(model_weights_path, map_location=device))
cnn = resnet50.to(device)

# Summary
summary(cnn, (3, 64, 64))


LymphoMNIST v0.0.1 @ https://github.com/Khayrulbuet13/Lympho3-MNIST.git
Dataset already exists. Skipping download.
Dataset already exists. Skipping download.


2024-03-10 21:27:55,745 - [INFO] - Train samples=40800, Validation samples=5100, Test samples=5100
/home/mdi220/.virtualenvs/tvsb/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mdi220/.virtualenvs/tvsb/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 32, 32]           9,408
       BatchNorm2d-2           [-1, 64, 32, 32]             128
              ReLU-3           [-1, 64, 32, 32]               0
         MaxPool2d-4           [-1, 64, 16, 16]               0
            Conv2d-5           [-1, 64, 16, 16]           4,096
       BatchNorm2d-6           [-1, 64, 16, 16]             128
              ReLU-7           [-1, 64, 16, 16]               0
            Conv2d-8           [-1, 64, 16, 16]          36,864
       BatchNorm2d-9           [-1, 64, 16, 16]             128
             ReLU-10           [-1, 64, 16, 16]               0
           Conv2d-11          [-1, 256, 16, 16]          16,384
      BatchNorm2d-12          [-1, 256, 16, 16]             512
           Conv2d-13          [-1, 256, 16, 16]          16,384
      BatchNorm2d-14          [-1, 256,

In [2]:
y_pred = []
y_true = []

with torch.no_grad():
    for image, target in train_dl:
        image, target = image.to(device), target.to(device)
        outputs = cnn(image)
        output = (torch.max(outputs, 1)[1]).data.cpu().numpy()
        y_pred.extend(output) # Save Prediction
        target = target.data.cpu().numpy()
        y_true.extend(target) # Save target
        
# Calculate accuracy
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

accuracy = np.trace(cm) / np.sum(cm)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.96


In [3]:
y_pred = []
y_true = []

with torch.no_grad():
    for image, target in test_dl:
        image, target = image.to(device), target.to(device)
        outputs = cnn(image)
        output = (torch.max(outputs, 1)[1]).data.cpu().numpy()
        y_pred.extend(output) # Save Prediction
        target = target.data.cpu().numpy()
        y_true.extend(target) # Save target
        
# Calculate accuracy
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

accuracy = np.trace(cm) / np.sum(cm)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.94


In [ ]:

project = Project()

logging.info(f'Using device={device} 🚀')

model_name = params['model']

# # define our comet experiment
# experiment = Experiment(
#     api_key="2iwTpjYhUb3dGr4yIiVtt1oRA",
#     project_name="TvsB-ablation",
#     # project_name="",
#     workspace="khayrulbuet13")

# experiment.log_parameters(params)
# experiment.set_name(model_name)




# define custom optimizer and instantiace the trainer `Model`
optimizer = optim.Adam(cnn.parameters(), lr=params['lr'])
model = Model(cnn, optimizer, "cross_entropy",
                batch_metrics=["accuracy"]).to(device)
# usually you want to reduce the lr on plateau and store the best model
# callbacks = [
#     ReduceLROnPlateau(monitor="val_acc", patience=100, verbose=True),
#     ModelCheckpoint(str(project.checkpoint_dir /
#                         f"""{datetime.datetime.now().strftime('%d %B %H:%M')}-model{model_name}.pt"""), save_best_only="True", verbose=True),
#     EarlyStopping(monitor="val_acc", patience=100, mode='max'),
#     CometCallback(experiment)
# ]

# model.fit_generator(
#     train_dl,
#     val_dl,
#     epochs=params['epochs'],
#     callbacks=callbacks,
# )


# # Save the final model weights after training with Poutyne
# model_weights_path = os.path.join(project.checkpoint_dir, f"{datetime.datetime.now().strftime('%d %B %H:%M')}_{model_name}_final_weights.pt")
# model.save_weights(model_weights_path)
# logging.info(f"Model weights saved to {model_weights_path}")


from sklearn.metrics import confusion_matrix
import numpy as np



def calculate_accuracy_poutyne(model, dataloader, device):
    # Ensure the model is in evaluation mode
    # model.model.eval()  # Note: Poutyne models wrap the PyTorch model in a .model attribute
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():  # Disable gradient computation
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            # Use predict_on_batch for predictions with Poutyne
            outputs = model.predict_on_batch(inputs)
            
            preds = np.argmax(outputs, axis=1)  # Use np.argmax to get the index of the max log-probability
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())
    
    # Calculate accuracy
    correct = sum(p == t for p, t in zip(all_preds, all_targets))
    accuracy = correct / len(all_preds)
    return accuracy


# Calculate the confusion matrices for the train and test sets
train_cm = calculate_accuracy_poutyne(model, train_dl, device)
test_cm = calculate_accuracy_poutyne(model, test_dl, device)




print("Train Confusion Matrix:",train_cm)
print("Test Confusion Matrix:",test_cm)



In [ ]:


def calculate_accuracy_poutyne(model, dataloader, device):
    # Ensure the model is in evaluation mode
    # model.model.eval()  # Note: Poutyne models wrap the PyTorch model in a .model attribute
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():  # Disable gradient computation
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            # Use predict_on_batch for predictions with Poutyne
            outputs = model.predict_on_batch(inputs)
            
            preds = np.argmax(outputs, axis=1)  # Use np.argmax to get the index of the max log-probability
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())
    
    # Calculate accuracy
    correct = sum(p == t for p, t in zip(all_preds, all_targets))
    accuracy = correct / len(all_preds)
    return accuracy


# Calculate the confusion matrices for the train and test sets
train_cm = calculate_accuracy_poutyne(model, train_dl, device)
test_cm = calculate_accuracy_poutyne(model, test_dl, device)


print("Train Confusion Matrix:",train_cm)
print("Test Confusion Matrix:",test_cm)



In [ ]:

project = Project()

logging.info(f'Using device={device} 🚀')

model_name = params['model']




# define custom optimizer and instantiace the trainer `Model`
optimizer = optim.Adam(cnn.parameters(), lr=params['lr'])
model = Model(cnn, optimizer, "cross_entropy",
                batch_metrics=["accuracy"]).to(device)


from sklearn.metrics import confusion_matrix
import numpy as np



def calculate_accuracy_poutyne(model, dataloader, device):
    # Ensure the model is in evaluation mode
    # model.model.eval()  # Note: Poutyne models wrap the PyTorch model in a .model attribute
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():  # Disable gradient computation
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            # Use predict_on_batch for predictions with Poutyne
            outputs = model.predict_on_batch(inputs)
            
            preds = np.argmax(outputs, axis=1)  # Use np.argmax to get the index of the max log-probability
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())
    
    # Calculate accuracy
    correct = sum(p == t for p, t in zip(all_preds, all_targets))
    accuracy = correct / len(all_preds)
    return accuracy


# Calculate the confusion matrices for the train and test sets
train_cm = calculate_accuracy_poutyne(model, train_dl, device)
test_cm = calculate_accuracy_poutyne(model, test_dl, device)




print("Train Confusion Matrix:",train_cm)
print("Test Confusion Matrix:",test_cm)

